# MMSD text-only Bi-LSTM comparison baseline

Uses packed text sequences and real-token mean pooling, with seeds 42, 43, and 44. Existing root-level results describe the previous single-seed implementation; rerun for updated per-seed artifacts and summary.json.

In [1]:
%pip install tqdm==4.67.1


  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Configuration


In [2]:
import ast
import csv
import json
import os
import pickle
import platform
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
ASSET_ROOT = DATA_ROOT / 'raw/mmsd'
SEEDS = (42, 43, 44)
BATCH_SIZE = 32
EPOCHS = 8
PATIENCE = 5
LEARNING_RATE = 0.001
L2 = 1e-7
SEQ_LEN = 75
NUM_WORKERS = 4
RUN_ROOT = Path('/kaggle/working/runs/text_baseline_packed_multiseed')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


PyTorch: 2.11.0+cpu Device: cpu


## Data and embeddings


In [3]:
class VocabularyLoader(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Global references are not allowed in vocabularies')

def load_vocabulary(path):
    with path.open('rb') as stream:
        vocabulary = VocabularyLoader(stream, encoding='latin1').load()
    if not isinstance(vocabulary, dict) or not all(
        isinstance(key, str) and type(value) is int for key, value in vocabulary.items()):
        raise ValueError(f'Invalid vocabulary: {path}')
    return vocabulary

word_vocab = load_vocabulary(ASSET_ROOT / 'words/vocab')
word_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'words/vector', dtype=np.float32))
assert word_weights.shape == (len(word_vocab), 200)
assert word_vocab['<pad>'] == 0
blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour',
           'humor', 'jokes', 'irony', 'ironic', 'exgag'}

class TextDataset(Dataset):
    def __init__(self, split):
        self.rows = [ast.literal_eval(line) for line in
                     (ASSET_ROOT / f'text/{split}.txt').read_text().splitlines() if line.strip()]
        self.rows = [row for row in self.rows if not blocked.intersection(row[1].split())]
        self.tokens = torch.zeros((len(self.rows), SEQ_LEN), dtype=torch.long)
        self.labels = torch.tensor([row[-1] for row in self.rows], dtype=torch.float32)
        for index, row in enumerate(self.rows):
            ids = [word_vocab.get(token, word_vocab['<unk>']) for token in row[1].split()[:SEQ_LEN]]
            self.tokens[index, :len(ids)] = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.tokens[index], self.labels[index], str(self.rows[index][0])

train_data = TextDataset('train')
valid_data = TextDataset('valid2')
test_data = TextDataset('test2')
generator = torch.Generator().manual_seed(SEEDS[0])
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', generator=generator,
                          persistent_workers=NUM_WORKERS > 0)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                         pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
positives = train_data.labels.sum().item()
positive_weight = (len(train_data) - positives) / positives
print('Train / validation / test:', len(train_data), len(valid_data), len(test_data))
print('Positive-class loss weight:', positive_weight)


Train / validation / test: 19816 2410 2409
Positive-class loss weight: 1.292987734320759


## Text-only Bi-LSTM


In [4]:
class TextBiLSTMClassifier(nn.Module):
    def __init__(self, word_weights):
        super().__init__()
        self.word_embedding = nn.Embedding.from_pretrained(word_weights.clone(), freeze=False, padding_idx=0)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(200, 256, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 1))
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.08, a=-0.16, b=0.16)
                nn.init.constant_(module.bias, 0.01)
        for name, parameter in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(parameter)
            else:
                nn.init.zeros_(parameter)
                if 'bias_ih' in name:
                    with torch.no_grad():
                        parameter[256:512].fill_(1.0)

    def forward(self, tokens):
        lengths = tokens.ne(0).sum(dim=1)
        if (lengths == 0).any():
            raise ValueError('Encountered a text sequence containing only padding')
        embedded = self.dropout(self.word_embedding(tokens))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        text_raw, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=tokens.size(1))
        mask = (torch.arange(tokens.size(1), device=tokens.device)[None, :] < lengths[:, None]).unsqueeze(-1)
        text_vector = (text_raw * mask).sum(dim=1) / lengths.unsqueeze(1)
        return self.classifier(text_vector).squeeze(-1)

    def regularization(self):
        return 0.5 * sum(parameter.square().sum() for name, parameter in self.named_parameters()
                         if parameter.requires_grad and name != 'word_embedding.weight')

@torch.no_grad()
def check_padding(model):
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    tokens = torch.tensor([[1, 2, 0], [3, 0, 0]], device=device)
    padded = torch.cat([tokens, tokens.new_zeros(2, 4)], dim=1)
    torch.testing.assert_close(model(tokens), model(padded))
    try:
        model(torch.zeros_like(tokens))
    except ValueError:
        pass
    else:
        raise AssertionError('All-padding text must be rejected')
    model.train(was_training)


## Evaluation


In [5]:
def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    predicted = np.asarray(probabilities) > 0.5
    tp = int(((labels == 1) & predicted).sum())
    fp = int(((labels == 0) & predicted).sum())
    tn = int(((labels == 0) & ~predicted).sum())
    fn = int(((labels == 1) & ~predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': (tp + tn) / len(labels), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}

@torch.no_grad()
def evaluate(network, loader):
    network.eval()
    labels, probabilities, tweet_ids = [], [], []
    for tokens, targets, ids in tqdm(loader, desc='Evaluate', leave=False):
        logits = network(tokens.to(DEVICE, non_blocking=True))
        probabilities.extend(logits.sigmoid().cpu().tolist())
        labels.extend(targets.tolist())
        tweet_ids.extend(ids)
    return binary_metrics(labels, probabilities), tweet_ids, labels, probabilities


## Training


In [6]:
def train_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    train_loader.generator.manual_seed(seed)
    run_dir = RUN_ROOT / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=False)
    config = {'model': 'text', 'variant': 'packed_bilstm', 'seed': seed,
              'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS, 'patience': PATIENCE,
              'learning_rate': LEARNING_RATE, 'lr_decay_per_epoch': 0.8,
              'l2': L2, 'sequence_length': SEQ_LEN, 'text_padding': 'packed_lstm_masked_mean',
              'hidden_size': 256, 'input_dropout': 0.2, 'gradient_clipping': 5.0,
              'positive_weight': positive_weight, 'checkpoint_selection': 'validation positive-class F1',
              'asset_root': str(ASSET_ROOT),
              'split_counts': {'train': len(train_data), 'valid': len(valid_data), 'test': len(test_data)},
              'upstream_revision': 'f42b16510208624d91fa545ca9bb64c6335f971e',
              'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__,
              'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
              'gpu_names': [torch.cuda.get_device_name(DEVICE)] if DEVICE.type == 'cuda' else [],
              'device': str(DEVICE), 'num_workers': NUM_WORKERS,
              'persistent_workers': NUM_WORKERS > 0, 'non_blocking_transfers': True}
    (run_dir / 'config.json').write_text(json.dumps(config, indent=2) + '\n')
    base_model = TextBiLSTMClassifier(word_weights).to(DEVICE)
    check_padding(base_model)
    network = base_model
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    optimizer = torch.optim.Adam([p for p in base_model.parameters() if p.requires_grad], lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)
    best_f1, stale_epochs = -1.0, 0
    history = []
    start = time.perf_counter()
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)
    for epoch in range(1, EPOCHS + 1):
        network.train()
        epoch_loss, seen = 0.0, 0
        epoch_start = time.perf_counter()
        progress = tqdm(train_loader, desc=f'Seed {seed} epoch {epoch}/{EPOCHS}', unit='batch')
        for tokens, labels, _ in progress:
            tokens, labels = tokens.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(network(tokens), labels) + L2 * base_model.regularization()
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(base_model.parameters(), 5.0)
            optimizer.step()
            epoch_loss += loss.item() * len(labels)
            seen += len(labels)
            progress.set_postfix(loss=epoch_loss / seen)
        valid_metrics, _, _, _ = evaluate(network, valid_loader)
        record = {'epoch': epoch, 'train_loss': epoch_loss / seen, 'validation': valid_metrics,
                  'learning_rate': optimizer.param_groups[0]['lr'],
                  'seconds': time.perf_counter() - epoch_start}
        history.append(record)
        (run_dir / 'history.json').write_text(json.dumps(history, indent=2) + '\n')
        print(json.dumps({'seed': seed, **record}))
        if valid_metrics['f1'] > best_f1:
            best_f1, stale_epochs = valid_metrics['f1'], 0
            temporary = run_dir / 'best.pt.tmp'
            torch.save({'model': base_model.state_dict(), 'epoch': epoch, 'validation_f1': best_f1}, temporary)
            os.replace(temporary, run_dir / 'best.pt')
        else:
            stale_epochs += 1
        scheduler.step()
        if stale_epochs >= PATIENCE:
            break
    training_seconds = time.perf_counter() - start
    peak_memory = [torch.cuda.max_memory_allocated(DEVICE)] if DEVICE.type == 'cuda' else []
    checkpoint = torch.load(run_dir / 'best.pt', map_location=DEVICE, weights_only=True)
    base_model.load_state_dict(checkpoint['model'])
    test_start = time.perf_counter()
    test_metrics, ids, labels, probabilities = evaluate(network, test_loader)
    test_seconds = time.perf_counter() - test_start
    results = {'model': 'text', 'seed': seed, 'selected_epoch': checkpoint['epoch'],
               'best_validation_f1': checkpoint['validation_f1'], 'test': test_metrics,
               'training_seconds_including_validation': training_seconds,
               'test_seconds_including_data_loading': test_seconds,
               'peak_allocated_gpu_bytes_per_device': peak_memory,
               'trainable_parameters': sum(p.numel() for p in base_model.parameters() if p.requires_grad)}
    (run_dir / 'results.json').write_text(json.dumps(results, indent=2) + '\n')
    with (run_dir / 'test_predictions.csv').open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['tweet_id', 'label', 'probability', 'prediction'])
        writer.writerows((tweet_id, int(label), probability, int(probability > 0.5))
                         for tweet_id, label, probability in zip(ids, labels, probabilities))
    print(json.dumps(results, indent=2))
    return results


## Run all seeds and save aggregate results


In [7]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
all_results = []
for seed in SEEDS:
    all_results.append(train_seed(seed))
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
summary = {metric: {'mean': float(np.mean([r['test'][metric] for r in all_results])),
                    'std': float(np.std([r['test'][metric] for r in all_results], ddof=1))}
           for metric in ('precision', 'recall', 'f1', 'accuracy')}
summary['seeds'] = all_results
(RUN_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))


Seed 42 epoch 1/8:   0%|          | 0/619 [00:00<?, ?batch/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"seed": 42, "epoch": 1, "train_loss": 0.4708175616089092, "validation": {"precision": 0.7958236658932715, "recall": 0.7153284671532847, "f1": 0.7534321801208127, "accuracy": 0.8136929460580913, "tp": 686, "fp": 176, "tn": 1275, "fn": 273}, "learning_rate": 0.001, "seconds": 94.361716935}


Seed 42 epoch 2/8:   0%|          | 0/619 [00:00<?, ?batch/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"seed": 42, "epoch": 2, "train_loss": 0.3130055410555761, "validation": {"precision": 0.7235915492957746, "recall": 0.8571428571428571, "f1": 0.7847255369928401, "accuracy": 0.8128630705394191, "tp": 822, "fp": 314, "tn": 1137, "fn": 137}, "learning_rate": 0.0008, "seconds": 89.10016300799992}


... (59 outputs left out)

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"seed": 44, "epoch": 8, "train_loss": 0.0709614653160242, "validation": {"precision": 0.7683786505538771, "recall": 0.7956204379562044, "f1": 0.7817622950819673, "accuracy": 0.8232365145228215, "tp": 763, "fp": 230, "tn": 1221, "fn": 196}, "learning_rate": 0.00020971520000000012, "seconds": 87.42315246099997}


Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{
  "model": "text",
  "seed": 44,
  "selected_epoch": 3,
  "best_validation_f1": 0.7900249376558603,
  "test": {
    "precision": 0.7372727272727273,
    "recall": 0.8456725755995829,
    "f1": 0.7877610490529383,
    "accuracy": 0.8185969281859693,
    "tp": 811,
    "fp": 289,
    "tn": 1161,
    "fn": 148
  },
  "training_seconds_including_validation": 689.898778002,
  "test_seconds_including_data_loading": 1.8986293050002132,
  "peak_allocated_gpu_bytes_per_device": [],
  "trainable_parameters": 3525569
}
{
  "precision": {
    "mean": 0.7259933754482392,
    "std": 0.011187061204698224
  },
  "recall": {
    "mean": 0.8519290928050053,
    "std": 0.011751228018336488
  },
  "f1": {
    "mean": 0.7838342695145771,
    "std": 0.003589230421217669
  },
  "accuracy": {
    "mean": 0.8129237581292376,
    "std": 0.0050215157912339905
  },
  "seeds": [
    {
      "model": "text",
      "seed": 42,
      "selected_epoch": 2,
      "best_validation_f1": 0.7847255369928401,
      "test":